In [1]:
import os
import whisperx
import torch
import imageio_ffmpeg
os.environ["PATH"] += os.pathsep + os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())

def get_parent_directory() -> str:
    """Get the parent directory for handling csv files.

    Returns:
        string: the path to the directory where directories for csv files are located
    """
    #create relative path for parent
    relative_parent = os.path.join(os.getcwd(), '..')

    #use abspath for absolute parent path
    return str(os.path.abspath(relative_parent)).replace('\\', '/')

directory = get_parent_directory()

In [2]:
with open(f'{directory}/env/whisperx_token.txt') as f:
    whisperx_token = f.read()

In [3]:
video_file_path = f'{directory}/input_videos/annin_lempielain_on_pupu.mp4'
output_file_path = f'{directory}/output_texts/annin_lempielain_on_pupu.txt'

In [4]:
video_file_path

'c:/Users/OMISTAJA/python_projects/Finnish_ASR_testing/input_videos/annin_lempielain_on_pupu.mp4'

In [ ]:
# Transcribe using Finnish language settings
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 4  
compute_type = "float16" if device == "cuda" else "int8"
audio = whisperx.load_audio(video_file_path)

In [ ]:
# Align timestamps
model_a, metadata = whisperx.load_align_model(language_code="fi", device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

In [ ]:
# Diarise speakers
diarize_model = whisperx.DiarizationPipeline(use_auth_token=hf_token, device=device)
diarize_segments = diarize_model(audio)

In [ ]:
# Merge text transcription with speaker IDs
final_result = whisperx.assign_word_speakers(diarize_segments, result)

In [ ]:
# Save the final file
output_file_path = "annin_lempielain_on_pupu.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for segment in final_result["segments"]:
        speaker = segment.get("speaker", "UNKNOWN_SPEAKER")
        text = segment["text"].strip()
        start = segment["start"]
        end = segment["end"]
        f.write(f"[{start:05.1f}s - {end:05.1f}s] {speaker}: {text}\n")